In [1]:
# noinspection stub-packages-advertiser
import json
import os
import sqlite3

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

DB_PATH = "../data/pubchemqc.db"
PAC5_DB_PATH = "../data/pac5.db"
PUBCHEMQC_DIRECTORY = r"E:\pubchemqc"
B3LYP_DIRECTORY = r"E:\b3lyppm6_redo\outputs"
HARTREE_TO_ELECTRONVOLT = 27.211386245981
EV_TO_KJ_MOL = 96.4853321233100184


def load_b3lyp_energy(pubchem_cid: int) -> float | None:
    """
    Load the total energy of the regenerated B3LYP/6-31G* single point of 1 molecule.

    The energy is the ``HF`` entry of the last archive block of ``{pubchem_cid}.out`` in `B3LYP_DIRECTORY`, converted
    from hartree to eV.

    Parameters
    ----------
    pubchem_cid : int
        PubChem CID of the molecule.

    Returns
    -------
    float or None
        Total energy in eV, or None when the output file or its archive block does not exist.
    """
    path = os.path.join(B3LYP_DIRECTORY, f"{pubchem_cid}.out")
    if not os.path.exists(path):
        return None
    file_size = os.path.getsize(path)
    with open(path, "rb") as handle:
        handle.seek(max(0, file_size - 65536))
        tail = handle.read().decode("ascii", "ignore")
    start = tail.rfind(" 1\\1\\")
    if start == -1:
        return None
    # Archive lines wrap at 70 characters; continuation lines start with 1 space
    archive = tail[start:].replace("\r", "").replace("\n ", "")
    position = archive.find("\\HF=")
    if position == -1:
        return None
    return float(archive[position + 4 :].split("\\")[0]) * HARTREE_TO_ELECTRONVOLT


def load_pubchemqc_energy(pubchem_cid: int) -> float | None:
    """
    Load the B3LYP/6-31G* total energy of the PubChemQC record of 1 molecule.

    Parameters
    ----------
    pubchem_cid : int
        PubChem CID of the molecule.

    Returns
    -------
    float or None
        The ``total-energy`` field of ``{pubchem_cid}.json`` in `PUBCHEMQC_DIRECTORY`, in eV, or None when the record
        does not have this field.
    """
    with open(os.path.join(PUBCHEMQC_DIRECTORY, f"{pubchem_cid}.json"), "rb") as handle:
        return json.loads(handle.read()).get("total-energy")


connection = sqlite3.connect(DB_PATH)
connection.execute(f"ATTACH DATABASE '{PAC5_DB_PATH}' AS pac5")

query = """
SELECT
    pubchemqc.pubchem_cid,
    pubchemqc.num_atoms,
    pubchemqc.pubchemqc_hl_gap,
    pubchemqc.b3lyppm6_hl_gap,
    pubchemqc.b3lyppm6_status,
    structures.num_rotatable_bonds
FROM
    pubchemqc
    LEFT JOIN pac5.pac5 AS structures ON structures.pubchem_cid = pubchemqc.pubchem_cid
WHERE
    pubchemqc.serialization IS NOT NULL
    AND pubchemqc.pubchemqc_hl_gap IS NOT NULL
    AND pubchemqc.b3lyppm6_hl_gap IS NOT NULL
ORDER BY
    pubchemqc.pubchem_cid
"""

molecules = pd.read_sql_query(query, connection)
connection.close()

has_output = molecules["b3lyppm6_status"] == "normal"
molecules["b3lyppm6_energy"] = [
    load_b3lyp_energy(pubchem_cid) if output else None
    for pubchem_cid, output in zip(molecules["pubchem_cid"], has_output)
]
molecules["pubchemqc_energy"] = [
    load_pubchemqc_energy(pubchem_cid) if output else None
    for pubchem_cid, output in zip(molecules["pubchem_cid"], has_output)
]
# E(PubChemQC) - E(regenerated), sign convention of Fig. 3: positive when regenerated geometry is lower in energy
molecules["energy_difference"] = (
    molecules["pubchemqc_energy"] - molecules["b3lyppm6_energy"]
) * EV_TO_KJ_MOL
energies = molecules.dropna(subset=["energy_difference"])
mean_energy_difference = energies["energy_difference"].mean()
energy_span = energies["energy_difference"].max() - energies["energy_difference"].min()
min_energy = energies["energy_difference"].min() - 0.05 * energy_span
max_energy = energies["energy_difference"].max() + 0.15 * energy_span

correlation = pearsonr(molecules["pubchemqc_hl_gap"], molecules["b3lyppm6_hl_gap"])[0]

print(f"Molecules with both HL-gaps: {len(molecules):,}")
print(f"Molecules with both total energies: {len(energies):,}")
print(f"Pearson Correlation (regenerated vs PubChemQC HL-gap): {correlation:.4f}")

fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.1,
)

fig.add_trace(
    go.Scattergl(
        x=energies["num_atoms"],
        y=energies["energy_difference"],
        mode="markers",
        name="Δ energy",
        text=energies["pubchem_cid"],
        marker=dict(color="#636EFA", opacity=0.6),
        hovertemplate="CID %{text}<br>Atoms: %{x}<br>Δ energy: %{y:.2f} kJ/mol<extra></extra>",
    ),
    row=1,
    col=1,
)

# Baseline at 0 and mean line
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_hline(
    y=mean_energy_difference, line_dash="solid", line_color="#636EFA", row=1, col=1
)

min_gap = min(molecules[["pubchemqc_hl_gap", "b3lyppm6_hl_gap"]].min()) - 0.5
max_gap = max(molecules[["pubchemqc_hl_gap", "b3lyppm6_hl_gap"]].max()) + 0.5

fig.add_trace(
    go.Scattergl(
        x=molecules["pubchemqc_hl_gap"],
        y=molecules["b3lyppm6_hl_gap"],
        mode="markers",
        name="HL-Gap",
        text=molecules["pubchem_cid"],
        marker=dict(color="#EF553B", size=8, opacity=0.7),
        hovertemplate="CID %{text}<br>PubChemQC: %{x:.3f} eV<br>Regenerated: %{y:.3f} eV<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=[min_gap, max_gap],
        y=[min_gap, max_gap],
        mode="lines",
        line=dict(dash="dash", color="black", width=2),
        name="Parity Line",
        showlegend=False,
    ),
    row=1,
    col=2,
)

for column, letter in [(1, "a"), (2, "b")]:
    fig.add_annotation(
        text=f"<b>{letter}</b>",
        xref="x domain",
        yref="y domain",
        x=0.03,
        y=0.98,
        xanchor="left",
        yanchor="top",
        showarrow=False,
        font=dict(size=28, color="black"),
        row=1,
        col=column,
    )

# Upper favored annotation
fig.add_annotation(
    text="Regenerated",
    xref="x domain",
    yref="y domain",
    x=0.98,
    y=0.84,
    xanchor="right",
    yanchor="top",
    showarrow=False,
    font=dict(size=24, color="black"),
    row=1,
    col=1,
)

# Mean annotation
fig.add_annotation(
    text=f" Mean: {'+' if mean_energy_difference > 0 else ''}{mean_energy_difference:.2f} kJ/mol ",
    xref="x domain",
    yref="y domain",
    x=0.14,
    y=0.97,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    bgcolor="rgba(255, 255, 255, 0.8)",
    bordercolor="black",
    font=dict(size=24, color="black"),
    row=1,
    col=1,
)

# Lower favored annotation
fig.add_annotation(
    text="PubChemQC",
    xref="x domain",
    yref="y domain",
    x=0.98,
    y=0.02,
    xanchor="right",
    yanchor="bottom",
    showarrow=False,
    font=dict(size=24, color="black"),
    row=1,
    col=1,
)

fig.add_annotation(
    text=f" r = {correlation:.3f} ",
    xref="x domain",
    yref="y domain",
    x=0.14,
    y=0.97,
    xanchor="left",
    yanchor="top",
    showarrow=False,
    bgcolor="rgba(255, 255, 255, 0.8)",
    bordercolor="black",
    font=dict(size=24, color="black"),
    row=1,
    col=2,
)

fig.update_xaxes(
    title_text="<b>Number of atoms</b>",
    row=1,
    col=1,
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
)
fig.update_yaxes(
    title_text="<b>Δ energy (kJ/mol)</b>",
    row=1,
    col=1,
    range=[min_energy, max_energy],
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
)

fig.update_xaxes(
    title_text="<b>PubChemQC HL-Gap (eV)</b>",
    row=1,
    col=2,
    range=[min_gap, max_gap],
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
)
fig.update_yaxes(
    title_text="<b>Regenerated HL-Gap (eV)</b>",
    row=1,
    col=2,
    range=[min_gap, max_gap],
    linewidth=1,
    linecolor="black",
    mirror=True,
    ticks="inside",
    showline=True,
)

fig.update_layout(
    template="plotly",
    font=dict(family="Helvetica", size=16),
    width=1200,
    height=600,
    showlegend=False,
)
fig.write_image("../figures/si_note_7_pubchemqc_comparison.png", scale=2)
fig.show()

Molecules with both HL-gaps: 11,198
Molecules with both total energies: 11,198
Pearson Correlation (regenerated vs PubChemQC HL-gap): 0.9797
